In [31]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import matplotlib.pyplot as plt
import heapq
import seaborn as sns

from environment.environment import GraphWorldMFG_MultiGroup
from trainer.amid_trainer_graph import GraphEdgeMFG_Trainer
from solver.solver import solve_multigroup, GraphMFG_OMD_EdgeSolver_MultiGroup
from visualization.visualizationh import plot_heatmap, compute_exploitability_multigroup, plot_losses, plot_losses_line

In [32]:
import os
from pathlib import Path
import yaml
import numpy as np
import torch

# 1. Configuration Loader
def load_config(config_path="config.yaml"):
    """Load configuration safely from a YAML file relative to working directory."""
    notebook_dir = Path(os.getcwd())
    config_file = notebook_dir / config_path
    
    with open(config_file, 'r') as f:
        config = yaml.safe_load(f)
    return config

# 2. Graph Environment Factory Pattern
def create_graph_mfg_from_config(config):
    """Factory function initializing the Graph MFG environment directly from your config."""
    device = config.get("device", "cuda" if torch.cuda.is_available() else "cpu")
    print(f"Target Device: {device}")

    trainer_cfg = config["trainer"]
    
    # Extract Graph Configuration
    graph_cfg = config["graph"]
    num_nodes = graph_cfg["num_nodes"]
    
    # CRITICAL: Extract the list of lists matrix and convert to a PyTorch Tensor
    raw_matrix = graph_cfg["adjacency_matrix"]
    adjacency_matrix = torch.tensor(raw_matrix, dtype=torch.float32, device=device)
    
    # Process Group Data (Sinks and Sources are flat integers here, not grid tuples)
    groups = []
    for g in config["groups"]:
        groups.append({
            "source": int(g["source"]),
            "sink": int(g["sink"]),
            "mass": float(g["mass"])
        })
    
    solver_cfg = config["solver"]
    waiting_time = solver_cfg.get("waiting_time", 0)
    # Instantiate the Graph World Environment
    # (Matches your 'GraphWorldMFG_MultiGroup' class structure)
    env = GraphWorldMFG_MultiGroup(
        num_nodes=num_nodes,
        groups=groups,
        adjacency_matrix=adjacency_matrix,
        H = solver_cfg.get("H", None),
        device=device
    )
    
    # Create solvers for each group
    
    
    solvers = [
        GraphMFG_OMD_EdgeSolver_MultiGroup(
            env=env,
            group_idx=k,
            eta=solver_cfg["eta"],
            tau=solver_cfg["tau"],
            T=solver_cfg["T"],
            alpha=solver_cfg["alpha"],
            H=solver_cfg.get("H", None)
        )
        for k in range(env.K)
    ]

    trainer = GraphEdgeMFG_Trainer(env, solvers, leader_lr=trainer_cfg["leader_lr"])
    
    return env, solvers, trainer, config

In [56]:
config = load_config("config_graph.yaml")

env, solvers, trainer, config = create_graph_mfg_from_config(config)

# Training loop
theta_leader = torch.zeros((env.N, env.N), device=env.device)  # Initialize leader's strategy
theta_leader[1,2] = 5.0
theta_leader[2,1] = 5.0

Target Device: cpu


In [2]:
node_mass, edge_occ, final_flows, policies, W_cong_history, zeta_history = solve_multigroup(env, solvers, T=50, W_max=15, theta_leader=theta_leader)

NameError: name 'solve_multigroup' is not defined

In [58]:
exploitability, V_best, V_pi = solvers[0].compute_exploitability(W_cong_history, W_max=15, theta_leader=theta_leader)
exploitability

0.023435592651367188

In [59]:
edge_occ.sum(dim=0).sum(dim=-1)  # Total flow across all groups and time steps

tensor([[[0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 4.3452e-01, 5.6548e-01, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 4.3452e-01, 5.6548e-01, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 4.3452e-01, 5.6548e-01, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
         [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[0.0000e+00, 7.5001e-02, 5.6548e-01, 0.

In [1]:
final_flows

NameError: name 'final_flows' is not defined

In [11]:
a = trainer.compute_social_loss(final_flows, W_cong_history)
a

tensor(13.9242)

In [12]:
node_mass, edge_occ, final_flows, policy_sim, W_cong_history = env.simulate_forward_with_policy(zeta_history[49], W_max=15, theta_leader=theta_leader)

In [13]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 4.7057e-01, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 4.2425e-01, 5.2588e-02, 0.0000e+00],
        [9.2469e-14, 0.0000e+00, 5.2316e-01, 0.0000e+00],
        [1.4088e-07, 7.3483e-15, 4.2265e-01, 3.8761e-02],
        [1.9292e-09, 1.3146e-07, 0.0000e+00, 5.2588e-02],
        [2.2533e-06, 6.0784e-05, 5.0807e-15, 5.2588e-02],
        [2.6380e-08, 1.8962e-09, 8.8526e-08, 1.1602e-01],
        [2.9019e-05, 2.2532e-06, 3.0543e-05, 5.7575e-01],
        [3.9133e-09, 8.6567e-08, 1.8879e-09, 5.7655e-01],
        [1.1406e-06, 2.9108e-05, 1.6395e-06, 6.6277e-01],
        [2.5842e-09, 8.6152e-09, 9.0377e-08, 9.9994e-01],
        [1.370

In [14]:
a = trainer.compute_social_loss(final_flows, W_cong_history)
a

tensor(13.9251)

In [15]:
loss = 0
for i in range(env.H):
    cost = final_flows[i, 3] - 1
    loss = loss + cost
loss

tensor(-13.9251)

In [16]:
print("Starting Leader Infrastructure Optimization...")
print("-" * 50)
flows_previous = final_flows.detach()  # Detach to avoid backprop through the entire history
W_cong_history = W_cong_history.detach()  # Detach to avoid backprop
losses = []

epochs = 25
for epoch in range(1, epochs + 1):
    
    # Execute one optimization step
    social_loss, flows_new, W_cong_history_new, theta_leader_new = trainer.train_step(flows_previous, W_cong_history, epoch)
    
    # Pass the newly generated flows as the historical footprint for the next epoch
    flows_previous = flows_new.detach() 
    W_cong_history = W_cong_history_new.detach()
    theta_leader = theta_leader_new.detach()

    # Optional: Recompute or update W_cong_history based on flows_new if required, 
    # otherwise it continues to adapt based on the internal solve_multigroup loop.
    
    if epoch % 1 == 0:
        print(f"Epoch {epoch:02d}/{epochs} | Social Loss (Travel Time): {social_loss:.4f}")
    
    losses.append(social_loss)

print("-" * 50)
print("Training Complete!") 

Starting Leader Infrastructure Optimization...
--------------------------------------------------
Iteration 1: Leader theta_leader =
tensor([[2.4545, 2.4006, 2.3269, 2.4869],
        [2.4268, 2.4202, 2.3365, 2.5738],
        [2.5489, 2.4864, 2.4511, 2.3652],
        [2.5972, 2.5467, 2.5217, 2.4556]])
Exploitability: 1.4314966201782227
Epoch 01/25 | Social Loss (Travel Time): 16.6450
Iteration 2: Leader theta_leader =
tensor([[2.1428, 1.4466, 0.7053, 2.8757],
        [2.2530, 1.5211, 2.2528, 1.6418],
        [1.9999, 2.2116, 3.0919, 1.3315],
        [2.8793, 2.4521, 2.9911, 2.3327]])
Exploitability: 0.18388748168945312
Epoch 02/25 | Social Loss (Travel Time): 14.5146
Iteration 3: Leader theta_leader =
tensor([[1.3467e+00, 7.3290e-04, 1.1345e-01, 1.5947e+00],
        [4.7668e-01, 3.5802e-01, 1.3211e-01, 1.0019e-04],
        [2.7724e+00, 6.0051e-01, 1.8123e+00, 1.4996e-02],
        [3.0980e+00, 1.6296e+00, 4.2079e+00, 1.7373e+00]])
Exploitability: 2.068765640258789
Epoch 03/25 | Social Lo

In [17]:
theta_leader[0,1] = 0.0
theta_leader[1,3] = 0.0

In [26]:
theta_leader * env.A

tensor([[0., 0., 0., 0.],
        [5., 0., 5., 0.],
        [5., 5., 0., 0.],
        [0., 0., 0., 0.]])

In [19]:
node_mass, edge_occ, final_flows, policies, W_cong_history, zeta_history = solve_multigroup(env, solvers, T=50, W_max=15, theta_leader=theta_leader)

In [27]:
final_flows

tensor([[1.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 3.6527e-01, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 6.4904e-02, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 2.8491e-01, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 2.8491e-01, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.6395e-01],
        [0.0000e+00, 0.0000e+00, 1.5858e-30, 4.6755e-01],
        [0.0000e+00, 0.0000e+00, 2.0127e-30, 7.2562e-01],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.0000e+00, 0.0000e+00, 0.0000e+00, 1.0000e+00],
        [0.000

In [28]:
exploitability, V_best, V_pi = solvers[0].compute_exploitability(W_cong_history, W_max=15, theta_leader=theta_leader)
exploitability

0.006037712097167969

In [29]:
a = trainer.compute_social_loss(final_flows, W_cong_history)
a

tensor(11.6429)

In [30]:
losses

[tensor(16.6450),
 tensor(14.5146),
 tensor(13.9124),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429),
 tensor(11.6429)]

In [24]:
los_tens = torch.tensor(losses, dtype=torch.float32)

In [25]:
plot_losses(losses, title="Leader Optimization Loss Over Epochs", xlabel="Epoch", ylabel="Social Loss (Travel Time)")

AttributeError: 'list' object has no attribute 'items'

<Figure size 800x600 with 0 Axes>